# 02 — Statik INT8 QAT (PTQ ile aynı şema), BERT-Tiny / SST-2

**Proje:** Uç Cihazlar İçin FPGA Tabanlı BERT-Tiny Hızlandırıcısı (BIL 401)

FP32 baz modelden (`deneyler/fp32_baz/model`) başlayıp `yazilim/nicele_qat.py` ile sahte niceleme (fake-quant) eğitimi yapar.
Şema `nicele_ptq.py` ile aynıdır: ağırlık kanal başına simetrik INT8, aktivasyon tensör başına sabit ölçek, bias INT32.
Nihai sayılar sahte niceleme ile değil, `nicele_ptq.QLinear`'ın **tamsayı yoluyla** ölçülür. PTQ sütunu aynı kalibrasyonla aynı betik içinde yeniden hesaplanır.

> **Çalışma zamanı:** T4 GPU. Hedef süre ≈ 15 dk; betik süre bütçesini aşacaksa epoch sayısını kendisi azaltır.
> Önkoşul: `yazilim/` altındaki güncel dosyalar GitHub'a gönderilmiş olmalı **ya da** Drive'da `bert-tiny-fpga/yazilim/` içinde durmalı (aşağıda 2. bölüm).

## 0. Ayarlar

In [ ]:
REPO_URL   = "https://github.com/rumeysakolip/bert-tiny-fpga.git"
DRIVE_ROOT = "/content/drive/MyDrive/bert-tiny-fpga"
FP32_DIR   = f"{DRIVE_ROOT}/deneyler/fp32_baz"
OUT_ROOT   = f"{DRIVE_ROOT}/deneyler/int8_qat"

# --- PTQ koşunuzda kullandığınız kalibrasyon/şema ayarlarının AYNISI (PTQ sütunu bunlarla hesaplanır) ---
OBSERVER      = "minmax"      # "minmax" | "percentile"
PCT           = 99.99         # yalnızca percentile için
ASIMETRIK     = False         # True: asimetrik aktivasyon
N_KALIB       = 512
FP32_KATMANLAR = None         # örn. "classifier|pooler" (PTQ'da FP32 bıraktığınız Linear'lar)

# --- QAT ---
EPOCHS        = 3
LR            = 5e-5
OGREN_OLCEK   = False         # True: aktivasyon ölçeklerini de öğren (deneysel)
SEEDS         = [42]          # gürültüyü görmek için [42, 43, 44] (her biri ~ +4-5 dk)
SURE_BUTCESI_DK = 12          # yalnızca eğitim; aşılacaksa epoch otomatik azalır

DUMAN_TESTI   = False         # True: 1 epoch, 2000 cümle ile hızlı ortam kontrolü (~1-2 dk)

## 1. Drive'ı bağla ve cihazı kontrol et

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, torch
os.makedirs(OUT_ROOT, exist_ok=True)
assert os.path.isdir(f"{FP32_DIR}/model"), f"FP32 model bulunamadı: {FP32_DIR}/model (önce 01b not defterini çalıştırın)"
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU yok. Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU. CPU'da çok yavaş olur.")

## 2. Kodu indir ve ortamı kur
Depodan klonlar; ardından Drive'daki `bert-tiny-fpga/yazilim/*.py` varsa **depodakilerin üzerine yazar** (henüz push edilmemiş dosyalar için).

In [ ]:
%cd /content
if not os.path.exists("/content/bert-tiny-fpga"):
    !git clone {REPO_URL}
else:
    !git -C /content/bert-tiny-fpga pull
%cd /content/bert-tiny-fpga
import glob, shutil
for p in glob.glob(f"{DRIVE_ROOT}/yazilim/*.py"):
    shutil.copy(p, "yazilim/"); print("Drive'dan kopyalandı:", os.path.basename(p))
for gerekli in ("nicele_qat.py", "nicele_ptq.py", "nicemleme_olcut.py", "egit_fp32.py", "sst2_veri.py"):
    assert os.path.exists(f"yazilim/{gerekli}"), f"yazilim/{gerekli} yok: depoya push edin veya Drive'daki yazilim/ klasörüne koyun"
!pip install -q -r requirements.txt
import transformers, datasets
print("PyTorch", torch.__version__, "| Transformers", transformers.__version__, "| Datasets", datasets.__version__)

## 3. QAT koşusu (PTQ referansı + QAT + tamsayı yolu değerlendirmesi)

In [ ]:
import subprocess, time, json
def kos(seed):
    ad = f"qat_lr{LR:g}_e{EPOCHS}_s{seed}" + ("_olcek" if OGREN_OLCEK else "") + ("_duman" if DUMAN_TESTI else "")
    run = f"{OUT_ROOT}/{ad}"; os.makedirs(run, exist_ok=True)
    cmd = ["python", "-u", "yazilim/nicele_qat.py",
           "--model_dir", f"{FP32_DIR}/model", "--fp32_csv", f"{FP32_DIR}/dogrulama_tahminleri.csv",
           "--out_dir", OUT_ROOT, "--ad", ad, "--seed", str(seed), "--kalib_seed", "42",
           "--observer", OBSERVER, "--pct", str(PCT), "--n_kalib", str(N_KALIB),
           "--epochs", str(1 if DUMAN_TESTI else EPOCHS), "--lr", str(LR),
           "--sure_butcesi_dk", str(SURE_BUTCESI_DK), "--agirlik_kaydet"]
    if ASIMETRIK: cmd.append("--asimetrik")
    if OGREN_OLCEK: cmd.append("--ogren_olcek")
    if FP32_KATMANLAR: cmd += ["--fp32_katmanlar", FP32_KATMANLAR]
    if DUMAN_TESTI: cmd += ["--max_train_samples", "2000"]
    t0 = time.time()
    with open(f"{run}/egitim_log.txt", "w", encoding="utf-8") as lf, \
         subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for satir in p.stdout:
            print(satir, end=""); lf.write(satir)
    assert p.returncode == 0, f"koşu başarısız (kod {p.returncode})"
    print(f"\n>>> seed {seed} bitti: {time.time()-t0:.0f} sn")
    return run

kosular = [kos(s) for s in SEEDS]

## 4. Sonuçlar: FP32 / PTQ / QAT yan yana

In [ ]:
from IPython.display import Markdown, display
R = [json.load(open(f"{r}/sonuclar.json", encoding="utf-8")) for r in kosular]
ptq, q0 = R[0]["ptq"], R[0]["qat"]
print(f"FP32: {q0['dogru_fp32']}/{q0['n']} (%{100*q0['dogruluk_fp32']:.2f}) | hedef ≥{q0['esikler']['min_dogru_hedef']}, üst sınır ≥{q0['esikler']['min_dogru_sinir']}")
print(f"Kullanılan epoch: {[r['kullanilan_epoch'] for r in R]} | eğitim süresi (sn): {[r['egitim_sn'] for r in R]}")
display(Markdown(open(f"{kosular[0]}/karsilastirma.md", encoding="utf-8").read()))
if len(R) > 1:
    print("\nTohum (seed) başına QAT:")
    for s, r in zip(SEEDS, R):
        q = r["qat"]; print(f"  seed {s}: {q['dogru_int8']}/{q['n']} | kayıp {q['kayip_puan']} puan ({q['kayip_cumle']} cümle) | uyuşma %{100*q['uyusma_orani']:.2f} | {q['karar']}")
    d = [r["qat"]["dogru_int8"] for r in R]
    print(f"  ortalama doğru: {sum(d)/len(d):.1f}, en kötü: {min(d)}, en iyi: {max(d)}  (PTQ: {ptq['dogru_int8']})")
print("\nTutarlılık (sahte niceleme ↔ tamsayı yolu):", R[0]["tutarlilik"])
print("Epoch geçmişi:", [(h['epoch'], round(h['train_loss'],4), round(100*h['val_acc_sahte_niceleme'],2)) for h in R[0]["gecmis"]])

**Yorum notu:** 872 cümlede 1 puan ≈ 8,72 cümle; 1–2 cümlelik farklar gürültüdür. Betik epoch seçmez (son epoch raporlanır).
Birden fazla LR/epoch denerseniz sonucu doğrulama kümesine bakarak seçmek iyimser olur; bunu raporda belirtin.

## 5. Drive'a yazımı tamamla

In [ ]:
drive.flush_and_unmount()
print("Drive senkronlandı:", OUT_ROOT)